# Evaluation framework (after Li, Ge & Jiang 2025, and ctSVGbench)
1. **Simulate** ctSVPs with known truth from real imaging data.
2. **Score** each method with EP, AUPRC, FDP@5%, type I error, and KS (`metrics.py`).
3. **Stress tests:** permutation null, rotation invariance, phenotype-label noise, spillover.

The anchor benchmark's full R pipeline (scDesign3 on Xenium → pseudo-spots) is in [weiqi-grace-li/ctsvg-benchmark](https://github.com/weiqi-grace-li/ctsvg-benchmark) (`simulators/realistic/`, `run/*.R`). Its intermediate data are on Zenodo record 21405877.

In [ ]:
import numpy as np, pandas as pd
from metrics import evaluate, rotate_coords, permute_within_type, jaccard_topk

## 1. A lightweight protein simulator (Python)
Start from real cells (coordinates + cell types) so locations and composition stay realistic, then generate marker values:
* **null markers**: cell-type mean + noise (constant within type, so composition-driven variation only),
* **ctSVPs**: planted spatial pattern **only inside the target cell type** (gradient, hotspot, or niche-dependent),
* optional **spillover**: mix a fraction of each neighbour's signal into the cell.

For a more realistic generator, fit scDesign3 (R, `family_use = "gaussian"`) per cell type to arcsinh intensities, as the benchmark does for Xenium.

In [ ]:
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(590)

def simulate(coords, ctype, n_null=30, n_sv=10, target_types=None, pattern="hotspot",
             effect=1.0, spill=0.0, k=6):
    n = len(ctype); types = np.unique(ctype)
    target_types = target_types or list(types[:2])
    mu = rng.normal(0, 1, size=(len(types), n_null + n_sv))          # cell-type means (lineage effects)
    X = mu[np.searchsorted(types, ctype)] + rng.normal(0, 1, size=(n, n_null + n_sv))
    xy = (coords - coords.min(0)) / np.ptp(coords, 0)
    truth = []
    for j in range(n_null, n_null + n_sv):
        t = rng.choice(target_types)
        if pattern == "gradient":
            f = xy[:, 0]
        else:
            c = rng.uniform(0.2, 0.8, 2); f = np.exp(-((xy - c) ** 2).sum(1) / 0.02)
        mask = ctype == t
        X[mask, j] += effect * (f[mask] - f[mask].mean()) / (f[mask].std() + 1e-9)
        truth.append((t, f"m{j}"))
    if spill > 0:
        _, ind = NearestNeighbors(n_neighbors=k + 1).fit(coords).kneighbors(coords)
        X = (1 - spill) * X + spill * X[ind[:, 1:]].mean(1)
    feats = [f"m{j}" for j in range(X.shape[1])]
    truth_set = set(truth)
    truth_df = pd.DataFrame([(ct, f, (ct, f) in truth_set) for ct in types for f in feats],
                            columns=["cell_type", "feature", "is_true"])
    return pd.DataFrame(X, columns=feats), truth_df

# Example with real coordinates/cell types from Squidpy IMC
# import squidpy as sq; a = sq.datasets.imc()
# X, truth = simulate(a.obsm["spatial"], a.obs["cell type"].to_numpy(), spill=0.1)

## 2. Run methods → long table `method, cell_type, feature, p_value` → evaluate
R methods write CSVs (see `01_ctSVG_methods/ctSVG_methods_tutorial.Rmd` §9); Python baselines likewise.

In [ ]:
# results = pd.concat([pd.read_csv(f) for f in ["results_ctsv_long.csv", "results_moran_within_type.csv"]])
# summary = evaluate(results, truth)
# summary.round(3)

## 3. Stress tests (ctSVGbench-style)
* **Permutation null**: shuffle coordinates within each cell type → every p-value should be Uniform(0,1) (type I error ≈ α, KS ≈ 0).
* **Rotation**: rotate by 30°, 60°, 90° → Jaccard of top-k calls should be ≈ 1 for a rotation-invariant method.
* **Label noise**: flip y% of cell-type labels → how fast does EP decay?

In [ ]:
# coords_perm = permute_within_type(coords, ctype, rng)
# coords_rot  = rotate_coords(coords, degrees=30)
# jaccard_topk(res_original, res_rotated, k=20)